# 01 — Data Inspection

Quick look at every data source: branch structure, event counts, basic distributions.
Run this first to verify your paths are correct and data is readable.

In [ ]:
import sys
sys.path.insert(0, '..')

from hibeam import config
from hibeam.plotting import style

cfg = config.load('../config.yaml')
style.apply(cfg)

print('Config loaded successfully.')
print('Simulation datasets:', list(cfg.paths.simulation.keys()))
print('Experimental runs:  ', list(cfg.paths.experimental.keys()))

## 1. Inspect experimental branch structure

Run this on any experimental ROOT file to see the exact branch names.
Useful when debugging data loading issues.

In [ ]:
from hibeam.io.exp_loader import inspect_branches

exp_path = config.resolve_path(cfg, cfg.paths.experimental.run_0042)
if exp_path.exists():
    inspect_branches(exp_path)
else:
    print(f'File not found: {exp_path}')
    print('Update paths.experimental.run_0042 in config.yaml')

## 2. Load and inspect simulation data

In [ ]:
from hibeam.io import sim_loader
import awkward as ak
import numpy as np

sim_path = config.resolve_path(cfg, cfg.paths.simulation.krakow)
if sim_path.exists():
    sim_data = sim_loader.load_prototpc(str(sim_path))
    sim_loader.edep_to_mev(sim_data)

    print(f"Events:        {sim_data['n_events']:,}")
    print(f"Edep type:     {sim_data['edep'].type}")
    print(f"Weights range: {sim_data['weights'].min():.3f} – {sim_data['weights'].max():.3f}")

    # Per-event totals
    sums = ak.to_numpy(ak.sum(sim_data['edep'], axis=1))
    nonzero = sums[sums > 0]
    print(f"Non-zero events: {len(nonzero):,} ({len(nonzero)/len(sums)*100:.1f}%)")
    print(f"Edep range: {nonzero.min():.4f} – {nonzero.max():.2f} MeV")
else:
    print(f'File not found: {sim_path}')

## 3. Quick raw histogram (no fitting)

In [ ]:
from hibeam.physics import dedx
from hibeam.plotting import histograms

values = dedx.compute_dedx(sim_data, method='sum_edep', min_steps=5)
print(f'{len(values):,} events after min_steps cut')

fig = histograms.plot_simple_histogram(
    values, bins=120,
    x_label=r'$\sum E_\mathrm{dep}$ [MeV]',
    title='Krakow simulation — raw Edep distribution',
    cfg=cfg,
)

## 4. Load experimental tracks

In [ ]:
from hibeam.io import exp_loader

exp_path = config.resolve_path(cfg, cfg.paths.experimental.run_0042)
if exp_path.exists():
    exp_data = exp_loader.load(
        str(exp_path),
        headers_dir=str(config.resolve_path(cfg, cfg.paths.headers_dir)),
        headers_so=str(config.resolve_path(cfg, cfg.paths.headers_so)),
        chi2_ndf_max=cfg.cuts.chi2_ndf_max,
        min_track_points=cfg.cuts.min_track_points,
    )

    print(f"dE/dx values: {len(exp_data['dedx']):,}")
    print(f"Tracks:       {exp_data['n_tracks']:,}")
    print(f"Events:       {exp_data['n_events']:,}")

    fig = histograms.plot_simple_histogram(
        exp_data['dedx'], bins=100,
        x_label='dE/dx [ADC/mm]',
        title=f"Experimental dE/dx — {exp_data['source']}",
        cfg=cfg,
    )
else:
    print(f'File not found: {exp_path}')

## 5. Load raw pad hits and display pad-plane

In [ ]:
from hibeam.plotting import displays

if exp_path.exists():
    raw = exp_loader.load_raw_hits(
        str(exp_path),
        noise_sigma=cfg.cuts.noise_sigma,
        entry_stop=500,
    )
    print(f"Loaded {raw['n_events']} events")

    displays.pad_plane_heatmap(
        raw['row'], raw['col'], raw['signal'],
        title=f"Pad-plane — {raw['source']}",
        cfg=cfg,
    )

## 6. Check segmentation data

In [ ]:
from hibeam.io import seg_loader

seg_dir = config.resolve_path(cfg, cfg.paths.segmentation.krakow_dir)
if seg_dir.is_dir():
    records = seg_loader.load(
        str(seg_dir),
        pattern=cfg.segmentation.pattern,
        min_steps=cfg.segmentation.min_steps,
    )
    print(f'\nLoaded {len(records)} segmentation files')
    for r in records[:5]:
        print(f"  nSec={r['nsec']:3d}: {r['n_total']:,} events, "
              f"mean nHits={r['mean_nhits_nonzero']:.1f}, "
              f"zero-hit={r['zero_frac']:.1%}")
    if len(records) > 5:
        print(f'  ... and {len(records)-5} more')
else:
    print(f'Directory not found: {seg_dir}')